# Writing to the Lakehouse

## Overview
- Files are stored in **OneLake** — Microsoft's unified storage layer
- Two areas inside a Lakehouse:
  - **Files/** — unmanaged files (CSV, Parquet, JSON, etc.)
  - **Tables/** — managed Delta tables (queryable via SQL endpoint)
- Writing to Tables creates Delta format automatically

## Setup — Create Schema and Load Flights to Delta Tables
This notebook uses the `writing_demo` schema to keep its tables isolated from other notebooks.
Narrowed to the Baton Rouge area airports (MSY/BTR) so writes, merges, and partitioning stay fast to run live.
Run these cells once before the rest of the demo.

In [ ]:
# Create an isolated schema for this demo
spark.sql("CREATE SCHEMA IF NOT EXISTS writing_demo")
print("Schema ready: writing_demo")

In [ ]:
from pyspark.sql.functions import col, year, to_date, monotonically_increasing_id

# Load all three years of flights, narrowed to the Baton Rouge area, with a stable row id for the merge demo later
(
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("Files/flights/flights_*.csv")
    .withColumn("FlightDate", to_date(col("FlightDate")))
    .withColumn("year", year(col("FlightDate")))
    .filter(col("Origin").isin("MSY", "BTR"))
    .withColumn("flight_id", monotonically_increasing_id())
    .write
    .mode("overwrite")
    .saveAsTable("writing_demo.flights")
)

print(f"flights table created: {spark.table('writing_demo.flights').count()} rows")

In [ ]:
# Load airlines.csv and save as a managed Delta table in writing_demo schema
(
    spark.read
    .format("csv")
    .option("header", "true")
    .load("Files/flights/airlines.csv")
    .write
    .mode("overwrite")
    .saveAsTable("writing_demo.airlines")
)

print(f"airlines table created: {spark.table('writing_demo.airlines').count()} rows")

## 1. Load some data to work with

In [ ]:
# Load Baton Rouge area flights from the writing_demo schema
flights = spark.table("writing_demo.flights")
display(flights)

## 2. Write to Files/ (unmanaged)

### Write as CSV

In [ ]:
# Write DataFrame to CSV in the Files area
# 'overwrite' replaces the output directory's contents if it already exists
(
    flights
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv("Files/Output/flights_export")
)

### Write as Parquet
- Parquet is a columnar format — more efficient for analytical queries
- Smaller file size than CSV, faster reads

In [ ]:
(
    flights
    .write
    .mode("overwrite")
    .parquet("Files/Output/flights_parquet")
)

## 3. Write to Tables/ (managed Delta)

### saveAsTable — creates a managed Delta table
- Shows up under **Tables** in the Lakehouse explorer
- Queryable from SQL endpoint, Power BI, and other notebooks

In [ ]:
# Write to a Delta table named 'flights_copy' in writing_demo schema
(
    flights
    .write
    .mode("overwrite")
    .saveAsTable("writing_demo.flights_copy")
)

In [ ]:
# Confirm it's there — read it back
df_check = spark.table("writing_demo.flights_copy")
df_check.count()

## 4. Write Modes
| Mode | Behavior |
|------|----------|
| `overwrite` | Replaces existing data |
| `append` | Adds rows to existing data |
| `ignore` | Does nothing if data exists |
| `error` | Throws error if data exists (default) |

### Append example — write 2024, then append 2023 on top

In [ ]:
from pyspark.sql.functions import col

# Filter to the most recent year
recent_flights = flights.filter(col("year") == 2024)

# First write — creates the table
(
    recent_flights
    .write
    .mode("overwrite")
    .saveAsTable("writing_demo.recent_flights")
)

print(f"Initial row count: {spark.table('writing_demo.recent_flights').count()}")

In [ ]:
# Append mode — adds rows without replacing
older_flights = flights.filter(col("year") == 2023)

(
    older_flights
    .write
    .mode("append")
    .saveAsTable("writing_demo.recent_flights")
)

print(f"After append row count: {spark.table('writing_demo.recent_flights').count()}")

## 5. Partitioning
- Splits the data into folders by a column value
- Speeds up queries that filter on the partition column
- Common to partition by date, year, or region

In [ ]:
# Write partitioned by year
# Creates a subfolder for each year: year=2022/, year=2023/, year=2024/
(
    flights
    .write
    .mode("overwrite")
    .partitionBy("year")
    .saveAsTable("writing_demo.flights_by_year")
)

In [ ]:
# Spark uses partition pruning — only reads the 2022 folder
spark.table("writing_demo.flights_by_year").filter(col("year") == 2022).count()

## 6. Upsert (Merge) with Delta Lake
- **Merge** = Insert new rows + Update existing rows in one operation
- Standard pattern for incremental loads

In [ ]:
from delta.tables import DeltaTable

# Imagine this is your incoming batch of new/updated records
incoming = flights.filter(col("year") >= 2023).limit(50)

# Reference the existing Delta table
target = DeltaTable.forName(spark, "writing_demo.flights_copy")

# Merge: update if flight_id matches, insert if it doesn't
(
    target.alias("target")
    .merge(
        incoming.alias("source"),
        "target.flight_id = source.flight_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

## 7. Delta Lake — Time Travel
- Delta keeps a transaction log of every write
- You can query any previous version

In [ ]:
# See the history of the table
target = DeltaTable.forName(spark, "writing_demo.flights_copy")
target.history().select("version", "timestamp", "operation").show(truncate=False)

In [ ]:
# Read version 0 — the original write before the merge
df_v0 = spark.read.format("delta").option("versionAsOf", 0).table("writing_demo.flights_copy")
print(f"Version 0 row count: {df_v0.count()}")
print(f"Current row count:   {spark.table('writing_demo.flights_copy').count()}")

---
**Next up:** `4. SemanticLink` — reading and writing the Power BI semantic model directly from a notebook.